# The Birthday Problem

This notebook works through one of the most fascinating problems in probability: in a group of random people, how likely is it that two of them share a birthday? The answer is famously non-intuitive — it takes only **23 people** for a shared birthday to be more likely than not. It is based on the seventh video of Week 1 (Introduction to Probability).


## 1. Introduction

With 3 random people, the probability that two share a birthday is clearly very low. With 200 people, it is clearly very high. But where is the tipping point? 30 people? 50? 100?

Surprisingly, in a party of just **30 friends** it is *more likely than not* that two people share a birthday — the probability of no shared birthday is only about 0.3, so there is roughly a **70% chance** of a match.

This problem is a beautiful application of two tools you already have:

- the **complement rule** ($P(\text{at least one match}) = 1 - P(\text{no match})$), and
- the **product rule** for independent events.

(For the whole notebook we assume a year has exactly 365 days — no February 29 birthdays.)


## 2. Theory

### Flip the question with the complement

Computing "at least two people share a birthday" directly is hard: a match could involve any pair, triple, and so on. It is far easier to compute the probability that **no two people share a birthday**, and then take the complement:

$$
P(\text{shared birthday}) = 1 - P(\text{all birthdays different})
$$

### Build up the group one person at a time

Track the probability that everyone so far has a different birthday, adding one person at a time:

- **Person 1** can have a birthday on any day: $\frac{365}{365} = 1$.
- **Person 2** must avoid 1 taken day: $\frac{364}{365}$.
- **Person 3** must avoid 2 taken days: $\frac{363}{365}$.
- **Person $k$** must avoid $k-1$ taken days: $\frac{365-(k-1)}{365}$.

Each new person's birthday is independent of the others, so the probabilities **multiply**:

$$
P(\text{no match among } n) = \frac{365}{365} \cdot \frac{364}{365} \cdot \frac{363}{365} \cdots \frac{365-n+1}{365}
$$

For $n = 366$ the product hits exactly 0: with more people than days, a repeat is guaranteed.


## 3. Worked Examples

### Example 1: A group of 9 people

$$
P(\text{no match}) = \frac{365}{365} \cdot \frac{364}{365} \cdots \frac{357}{365} \approx 0.905
$$

With nine people, there is roughly a 90% chance all birthdays are different — matches are still unlikely.

### Example 2: The party of 30

Multiplying the 30 factors gives

$$
P(\text{no match among } 30) \approx 0.294
$$

so $P(\text{shared birthday}) \approx 1 - 0.294 = 0.706$ — about **70%**.

### Example 3: The tipping point

Following the product as the group grows:

| People | P(no match) |
|---|---|
| 2 | 0.997 |
| 9 | 0.905 |
| 20 | 0.589 |
| **23** | **0.493** |
| 30 | 0.294 |
| 50 | 0.030 |
| 100 | ≈ 0.0000003 |
| 366 | 0 |

At **23 people** the probability of no match drops below one half — so a shared birthday becomes the better bet. If you're ever at a party with 23 or more people, bet on a repeated birthday: you'll win more than half the time.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)


### Computing the product


In [ ]:
def p_no_match(n, days=365):
    factors = (days - np.arange(n)) / days     # 365/365, 364/365, ...
    return factors.prod()

for n in [2, 9, 20, 23, 30, 50, 100]:
    p = p_no_match(n)
    print(f"n = {n:3d}:  P(no match) = {p:.6f}   P(shared) = {1 - p:.6f}")


### Finding the tipping point


In [ ]:
n = 1
while p_no_match(n) > 0.5:
    n += 1
print(f"P(no match) drops below 0.5 at n = {n} people "
      f"({p_no_match(n):.4f}; at {n - 1} it was {p_no_match(n - 1):.4f})")


### Checking with a simulation

We simulate many parties of 30 random birthdays and count how often at least two coincide.


In [ ]:
n_parties = 100_000
group_size = 30

birthdays = rng.integers(0, 365, size=(n_parties, group_size))
has_match = np.array([len(np.unique(party)) < group_size for party in birthdays])

print(f"Simulated P(shared birthday, n=30): {has_match.mean():.4f}")
print(f"Theoretical:                        {1 - p_no_match(30):.4f}")


## 5. Visualizations

### The probability curve

The video's key plot: number of people on the horizontal axis, probability of **no** repeated birthday on the vertical axis. The curve drops fast and crosses 0.5 at exactly 23 people.


In [ ]:
ns = np.arange(1, 101)
p_no = np.array([p_no_match(n) for n in ns])

plt.figure(figsize=(9, 5))
plt.plot(ns, p_no, label="P(no shared birthday)")
plt.plot(ns, 1 - p_no, label="P(at least one shared birthday)")
plt.axhline(0.5, color="gray", linestyle=":")
plt.axvline(23, color="red", linestyle="--", label="n = 23: the tipping point")
plt.title("The birthday problem: a match becomes likely surprisingly fast")
plt.xlabel("Number of people")
plt.ylabel("Probability")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


### Why intuition fails: count the pairs

The reason the answer feels wrong is that we instinctively think of *our own* birthday matching someone else's. But any pair can match — and the number of pairs grows much faster than the number of people: 23 people form $\binom{23}{2} = 253$ pairs.


In [ ]:
ns = np.arange(1, 61)
pairs = ns * (ns - 1) // 2

plt.figure(figsize=(9, 5))
plt.plot(ns, ns, label="People")
plt.plot(ns, pairs, label="Pairs of people")
plt.axvline(23, color="red", linestyle="--", label="23 people = 253 pairs")
plt.title("The number of possible matching pairs grows quadratically")
plt.xlabel("Number of people")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 6. Exercises

### Basic

1. Using the table above, what is the probability that at least two people share a birthday in a group of 20?
2. Why is $P(\text{no match}) = 0$ for a group of 366 people?

### Intermediate

3. Compute the probability that at least two people share a birthday in a group of 15.
4. In a group of 23 people, what is the probability that **you** (one fixed person) share a birthday with at least one of the other 22? Compare it with the ~50% group answer and explain the difference.

### Advanced

5. Suppose a planet's year has 500 days. How many people are needed before a shared birthday becomes more likely than not? Adapt the `p_no_match` function.


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  P(shared, n=20) = 1 - 0.589 =", round(1 - p_no_match(20), 3))

# Exercise 2
print("2.  With 366 people and 365 days, at least two birthdays must "
      "coincide (pigeonhole): the 366th factor is 0/365 = 0.")

# Exercise 3
print("3.  P(shared, n=15) =", round(1 - p_no_match(15), 4))

# Exercise 4: each of the 22 others misses your birthday with prob 364/365
p_you = 1 - (364 / 365) ** 22
print(f"4.  P(someone matches YOU) = 1 - (364/365)^22 = {p_you:.4f} — "
      "only ~6%, because this counts just 22 pairs, not all 253.")

# Exercise 5: 500-day year
n = 1
while p_no_match(n, days=500) > 0.5:
    n += 1
print(f"5.  With a 500-day year the tipping point is n = {n} people "
      f"(P(no match) = {p_no_match(n, days=500):.4f}).")


## 8. Conclusion

- The birthday problem is solved by combining the **complement rule** with the **product rule**: compute $P(\text{no match})$ as a product of shrinking fractions, then subtract from 1.
- The probability of a match crosses 50% at just **23 people**, and reaches ~97% at 50 people.
- Intuition fails because we think in terms of *our own* birthday (22 chances to match) instead of *all pairs* (253 chances at $n = 23$).
- The same build-it-up-one-person-at-a-time technique — multiply the probability that each new arrival avoids all previous choices — appears throughout probability and computer science (e.g. hash collisions).
